# VizEx — experimental analysis

Do the geometric features of stroboscopically induced visual hallucinations vary
with flicker frequency? Strobe trials only (`stimulus_type == LIGHT`).

Produces five small panels (one per feature) showing the condition mean with
±1 SD, a separate key, an omnibus table, and a planned-contrast table.

Requires `010_build_index` and `015_feature_extraction` to have been run first.

In [ ]:
## imports

import io
import textwrap
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

# Only silence library churn. A blanket ignore would also hide the
# "Random effects covariance is singular" warnings, which matter here.
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [ ]:
## paths and features

PROJECT_PATH = Path().resolve().parent
TABLES_PATH = PROJECT_PATH / "015_tables"
VIZ_PATH = PROJECT_PATH / "020_visualizations"
VIZ_PATH.mkdir(parents=True, exist_ok=True)

TRIALS_FEATURES_CSV = TABLES_PATH / "vizex_features.csv"

FEATURE_LABELS = {
    "detail": "detail level",
    "r_directionality": "radial orientation",
    "theta_directionality": "tangential orientation",
    "brightness": "brightness",
    "contrast": "contrast",
}
FEATURES = list(FEATURE_LABELS)

EXPERIMENT_LABEL = "Experiment 4.1 (drawings)"

In [ ]:
## load and prepare the strobe-trial frame

df_trials = pd.read_csv(TRIALS_FEATURES_CSV)
df_trials["condition_code"] = df_trials["condition_code"].astype(str)

uncropped = df_trials["png_filename"].str.contains("uncropped", na=False)
if uncropped.any():
    raise ValueError(f"{int(uncropped.sum())} uncropped rows present -- re-run 015.")

for feature in FEATURES:
    if feature not in df_trials.columns:
        raise KeyError(f"missing feature column '{feature}' -- re-run 015.")
    if df_trials[feature].isna().all():
        raise ValueError(f"feature column '{feature}' is all NaN -- re-run 015.")

df = df_trials[df_trials["stimulus_type"] == "LIGHT"].copy()

for col in FEATURES + ["condition_hz"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["condition_hz"] = df["condition_hz"].round(2)   # kill float noise before grouping
df = df.dropna(subset=FEATURES + ["condition_hz", "participant_code"]).copy()

# Ordered categorical over the frequencies actually present.
#
# NOTE: pd.Categorical(df["condition_hz"].sort_values()) would drop the index and
# assign POSITIONALLY, giving each row the i-th smallest frequency rather than its
# own. Always pass explicit categories.
HZ_LEVELS = sorted(df["condition_hz"].dropna().unique().tolist())
df["hz_cat"] = pd.Categorical(df["condition_hz"], categories=HZ_LEVELS, ordered=True)
assert (df["hz_cat"].astype(float) == df["condition_hz"]).all(), "hz_cat is misaligned"

print(f"{len(df):,} strobe trials, {df['participant_code'].nunique()} participants")
print(f"frequencies: {HZ_LEVELS}")
print(df.groupby("condition_hz").size().rename("n").to_string())

## Figures

Five small panels plus a separate key, to be assembled by hand.

In [ ]:
# =====================================================================
# FIGURE STYLE -- adjust these
# =====================================================================

FIG_SIZE = (4.6, 3.9)          # inches, exact canvas size of every saved panel
FIG_DPI = 300
SAVE_FORMATS = ("png", "svg")

# Axes rectangle as a fraction of the figure: (left, bottom, width, height).
# Fixed rather than auto-fitted, so all five panels come out at identical size and
# the plot areas line up when they are assembled by hand.
AXES_RECT = (0.22, 0.22, 0.72, 0.66)

AXIS_LABEL_WRAP = 28           # axis labels wrap at this many characters

FONT_SIZE_TITLE = 19
FONT_SIZE_AXIS_LABEL = 13
FONT_SIZE_TICK = 13
FONT_WEIGHT_TITLE = "bold"
FONT_WEIGHT_AXIS_LABEL = "bold"

LINE_WIDTH_SPINE = 2.8
LINE_WIDTH_TICK = 2.8
TICK_LENGTH = 6

LINE_WIDTH_MEAN = 4.0          # line joining the condition means
LINE_WIDTH_SD = 2.8            # standard-deviation whiskers
SD_CAP_SIZE = 6
SD_STYLE = "errorbar"          # "errorbar" (whiskers) or "band" (shaded ribbon)
SD_BAND_ALPHA = 0.18
SD_MULTIPLIER = 1.0            # 1.0 = +/- 1 SD

MARKER_SIZE_MEAN = 9
MARKER_EDGE_WIDTH = 2.5
MARKER_SIZE_DATA = 18          # individual trials
MARKER_ALPHA = 0.16
JITTER = 0.13

COLOR_POINTS = "#000000"
COLOR_MEAN = "#003366"
COLOR_SD = "#003366"

Y_TICK_COUNT = 4
X_LABEL = "stroboscopic frequency (Hz)"

TITLE_PAD = 10
LABEL_PAD = 7

# Manual y-axis upper bounds. None = autoscale from the data.
# High outliers otherwise squash the condition means onto the baseline.
Y_MAX_BY_FEATURE = {
    "detail": 0.75,
    "r_directionality": None,
    "theta_directionality": None,
    "brightness": None,
    "contrast": None,
}
# None = autoscale. 0.0 anchors the axis at zero.
Y_MIN_BY_FEATURE = {f: None for f in FEATURES}

# Key
KEY_FIG_SIZE = (7.6, 0.9)
KEY_FONT_SIZE = 14
KEY_NCOL = 3
KEY_LABELS = {
    "points": "individual trial",
    "mean": "condition mean",
    "sd": "\u00b1 1 SD",
}
# =====================================================================


def _save(fig, stem, tight=False):
    """Panels are saved at the exact figure size (no tight bbox) so every one has
    the same dimensions and the same axes position."""
    kw = dict(bbox_inches="tight", pad_inches=0.02) if tight else {}
    for ext in SAVE_FORMATS:
        fig.savefig(VIZ_PATH / f"{stem}.{ext}", dpi=FIG_DPI, transparent=False, **kw)
    print(f"saved: {stem}.{'/'.join(SAVE_FORMATS)}")


def _warn_if_clipped(fig, stem, rect_name="AXES_RECT"):
    """Renders the figure and checks whether any ink reaches the canvas edge.

    Because the axes rectangle is fixed, changing font sizes can push labels off
    the canvas. This says so explicitly instead of silently cropping them.
    """
    buf = io.BytesIO()
    fig.savefig(buf, dpi=100, format="png")
    buf.seek(0)
    a = np.asarray(Image.open(buf).convert("L"))
    edges = {
        "top": a[:2], "bottom": a[-2:], "left": a[:, :2], "right": a[:, -2:],
    }
    hit = [side for side, px in edges.items() if (px < 250).any()]
    if hit:
        fixes = {"left": f"increase {rect_name}[0]", "bottom": f"increase {rect_name}[1]",
                 "right": f"decrease {rect_name}[2]", "top": f"decrease {rect_name}[3]"}
        print(f"  WARNING [{stem}]: content is clipped at the "
              f"{', '.join(hit)} edge(s). Fix: "
              f"{'; '.join(fixes[s] for s in hit)}, or enlarge FIG_SIZE.")


def _wrap(text: str) -> str:
    return textwrap.fill(text, AXIS_LABEL_WRAP)


def _style_axes(ax):
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_linewidth(LINE_WIDTH_SPINE)
    ax.tick_params(axis="both", width=LINE_WIDTH_TICK, length=TICK_LENGTH,
                   labelsize=FONT_SIZE_TICK)
    ax.grid(False)


def fmt_hz(hz) -> str:
    """40.0 -> '40', 2.5 -> '2.5'. Plain numbers, no unit; the unit is on the axis."""
    hz = float(hz)
    return str(int(hz)) if hz.is_integer() else str(hz)


def plot_feature_vs_frequency(feature, data, *, hz_levels=None, save=True,
                              stem=None, y_max=None, y_min=None):
    """One small panel: faint jittered trials, condition mean, +/- 1 SD. No legend."""
    label = FEATURE_LABELS[feature]
    levels = list(hz_levels) if hz_levels is not None else list(data["hz_cat"].cat.categories)
    x_pos = np.arange(len(levels), dtype=float)
    rng = np.random.RandomState(42)

    y_max = Y_MAX_BY_FEATURE.get(feature) if y_max is None else y_max
    y_min = Y_MIN_BY_FEATURE.get(feature) if y_min is None else y_min

    grouped = data.groupby("condition_hz")[feature]
    means = grouped.mean().reindex(levels).to_numpy(float)
    sds = grouped.std(ddof=1).reindex(levels).to_numpy(float) * SD_MULTIPLIER
    sds = np.nan_to_num(sds, nan=0.0)

    fig = plt.figure(figsize=FIG_SIZE)
    ax = fig.add_axes(AXES_RECT)

    for i, hz in enumerate(levels):
        vals = data.loc[data["condition_hz"] == hz, feature].to_numpy(float)
        if vals.size:
            ax.scatter(x_pos[i] + rng.normal(0, JITTER, size=vals.size), vals,
                       s=MARKER_SIZE_DATA, color=COLOR_POINTS, alpha=MARKER_ALPHA,
                       edgecolors="none", zorder=1, clip_on=True)

    if SD_STYLE == "band":
        ax.fill_between(x_pos, means - sds, means + sds, color=COLOR_SD,
                        alpha=SD_BAND_ALPHA, linewidth=0, zorder=2)
    else:
        ax.errorbar(x_pos, means, yerr=sds, fmt="none", ecolor=COLOR_SD,
                    elinewidth=LINE_WIDTH_SD, capsize=SD_CAP_SIZE,
                    capthick=LINE_WIDTH_SD, zorder=2, clip_on=True)

    ax.plot(x_pos, means, color=COLOR_MEAN, linewidth=LINE_WIDTH_MEAN, marker="o",
            markersize=MARKER_SIZE_MEAN, markerfacecolor="white",
            markeredgewidth=MARKER_EDGE_WIDTH, markeredgecolor=COLOR_MEAN,
            solid_capstyle="round", zorder=3, clip_on=False)

    # y limits: manual bound where given, otherwise autoscale
    bottom, top = ax.get_ylim()
    bottom = bottom if y_min is None else float(y_min)
    top = top if y_max is None else float(y_max)
    ax.set_ylim(bottom, top)

    hidden = int((data[feature] > top).sum() + (data[feature] < bottom).sum())
    if hidden:
        print(f"  [{feature}] y-axis clipped to ({bottom:.3g}, {top:.3g}): "
              f"{hidden} of {len(data)} trials ({hidden / len(data):.1%}) fall outside "
              f"and are not drawn.")

    ax.set_xlim(-0.5, len(levels) - 0.5)
    ax.set_xticks(x_pos)
    ax.set_xticklabels([fmt_hz(hz) for hz in levels], rotation=0)
    ax.yaxis.set_major_locator(MaxNLocator(nbins=Y_TICK_COUNT))

    ax.set_title(label, fontsize=FONT_SIZE_TITLE, fontweight=FONT_WEIGHT_TITLE,
                 pad=TITLE_PAD)
    ax.set_xlabel(_wrap(X_LABEL), fontsize=FONT_SIZE_AXIS_LABEL,
                  fontweight=FONT_WEIGHT_AXIS_LABEL, labelpad=LABEL_PAD)
    ax.set_ylabel(_wrap(label), fontsize=FONT_SIZE_AXIS_LABEL,
                  fontweight=FONT_WEIGHT_AXIS_LABEL, labelpad=LABEL_PAD)
    _style_axes(ax)

    _warn_if_clipped(fig, stem or f"exp_{feature}")
    if save:
        _save(fig, stem or f"exp_{feature}")
    plt.show()
    plt.close(fig)


def make_experimental_key(*, save=True):
    """Standalone key for all five experimental panels."""
    sd_handle = (Patch(facecolor=COLOR_SD, alpha=SD_BAND_ALPHA, label=KEY_LABELS["sd"])
                 if SD_STYLE == "band"
                 else Line2D([], [], color=COLOR_SD, linewidth=LINE_WIDTH_SD,
                             marker="_", markersize=10, markeredgewidth=LINE_WIDTH_SD,
                             label=KEY_LABELS["sd"]))
    handles = [
        Line2D([], [], linestyle="none", marker="o",
               markersize=np.sqrt(MARKER_SIZE_DATA) * 1.9, color=COLOR_POINTS,
               alpha=min(MARKER_ALPHA * 3.0, 1.0), label=KEY_LABELS["points"]),
        Line2D([], [], color=COLOR_MEAN, linewidth=LINE_WIDTH_MEAN, marker="o",
               markersize=MARKER_SIZE_MEAN, markerfacecolor="white",
               markeredgewidth=MARKER_EDGE_WIDTH, markeredgecolor=COLOR_MEAN,
               label=KEY_LABELS["mean"]),
        sd_handle,
    ]
    fig, ax = plt.subplots(figsize=KEY_FIG_SIZE)
    ax.axis("off")
    ax.legend(handles=handles, loc="center", ncol=KEY_NCOL, frameon=False,
              fontsize=KEY_FONT_SIZE, handlelength=2.4, columnspacing=2.0,
              handletextpad=0.8, borderpad=0.0)
    if save:
        _save(fig, "exp_key", tight=True)
    plt.show()
    plt.close(fig)

In [ ]:
## draw all five panels and the key

for _feature in FEATURES:
    plot_feature_vs_frequency(_feature, df)

make_experimental_key()

## Models

In [ ]:
# =====================================================================
# MODEL AND TABLE SETTINGS -- adjust these
# =====================================================================

CORRECTION_METHOD = "holm"

DECIMALS_COEF = 6      # b, SE, z, residual variance, ICC
SCI_DECIMALS = 2       # p-values and random-effect variance

# ML (reml=False) matches Wald z tests on the fixed effects and reproduces the
# earlier 10 vs 15 Hz follow-up. REML gives less biased variance components but
# is not comparable across different fixed-effect structures.
USE_REML = False

# Planned contrasts. Each fits <feature> ~ C(group) with a random participant
# intercept; the reported b is (level - reference).
CONTRASTS = [
    {
        "label": "Experimental vs Control",
        "reference": "Control",
        "subset": lambda d: d[d["condition_type"].isin(["EXPERIMENTAL", "CONTROL"])],
        "group": lambda d: np.where(d["condition_type"].eq("EXPERIMENTAL"),
                                    "Experimental", "Control"),
    },
    {
        "label": "15 Hz vs 10 Hz",
        "reference": "10 Hz",
        "subset": lambda d: d[d["condition_hz"].isin([10, 15])],
        "group": lambda d: np.where(d["condition_hz"].eq(15), "15 Hz", "10 Hz"),
    },
]
# =====================================================================


def fit_mixedlm(formula, data, reml=USE_REML):
    """Random participant intercept, trying optimisers until one converges."""
    md = smf.mixedlm(formula, data, groups=data["participant_code"])
    last = None
    for method in ("lbfgs", "bfgs", "cg", "powell", "nm"):
        try:
            last = md.fit(reml=reml, method=method, maxiter=2000, disp=False)
            if getattr(last, "converged", False):
                return md, last
        except Exception:
            continue
    return md, last


def format_table(table, *, sci_cols, coef_cols):
    out = table.copy()
    for c in coef_cols:
        if c in out:
            out[c] = out[c].map(lambda v: "" if pd.isna(v) else f"{v:.{DECIMALS_COEF}f}")
    for c in sci_cols:
        if c in out:
            out[c] = out[c].map(lambda v: "" if pd.isna(v) else f"{v:.{SCI_DECIMALS}E}")
    return out

### Omnibus: does the feature vary across frequency at all?

In [ ]:
omnibus_rows = []
for feature in FEATURES:
    md, mdf = fit_mixedlm(f"{feature} ~ C(condition_hz)", df)
    if mdf is None:
        print(f"  WARNING [{feature}]: every optimiser failed; skipping.")
        continue

    try:
        t = mdf.wald_test_terms().table
        chi2 = float(np.squeeze(t.loc["C(condition_hz)", "statistic"]))
        dfc = int(np.squeeze(t.loc["C(condition_hz)", "df_constraint"]))
        p = float(np.squeeze(t.loc["C(condition_hz)", "pvalue"]))
    except Exception as e:
        print(f"  WARNING [{feature}]: Wald test failed ({e}); recording NaN.")
        chi2, dfc, p = np.nan, -1, np.nan

    re_var = float(mdf.cov_re.iloc[0, 0])
    resid_var = float(mdf.scale)
    mean = float(df[feature].mean())

    if re_var < 1e-10:
        print(f"  NOTE [{feature}]: random-effect variance ~0 ({re_var:.3g}); "
              f"the ICC is not identified.")

    omnibus_rows.append({
        "Experiment": EXPERIMENT_LABEL,
        "Feature": FEATURE_LABELS[feature],
        "Mean": mean,
        "Wald chi-square": chi2,
        "df": dfc,
        "p (uncorrected)": p,
        "Random-effect variance": re_var,
        "Residual variance": resid_var,
        "Trial-level SD": float(np.sqrt(resid_var)),
        "Trial-level CV (%)": (np.sqrt(resid_var) / mean * 100) if mean else np.nan,
        "Intraclass correlation coefficient": re_var / (re_var + resid_var),
        "n (trials)": int(len(df)),
        "n (participants)": int(df["participant_code"].nunique()),
    })

omnibus_raw = pd.DataFrame(omnibus_rows)
omnibus_raw["p (Holm-adjusted)"] = multipletests(
    omnibus_raw["p (uncorrected)"].fillna(1.0), alpha=0.05, method=CORRECTION_METHOD)[1]
omnibus_raw = omnibus_raw[[
    "Experiment", "Feature", "Mean", "Wald chi-square", "df",
    "p (uncorrected)", "p (Holm-adjusted)",
    "Random-effect variance", "Residual variance", "Trial-level SD",
    "Trial-level CV (%)", "Intraclass correlation coefficient",
    "n (trials)", "n (participants)"]]

omnibus_table = format_table(
    omnibus_raw,
    sci_cols=["p (uncorrected)", "p (Holm-adjusted)", "Random-effect variance"],
    coef_cols=["Mean", "Wald chi-square", "Residual variance", "Trial-level SD",
               "Trial-level CV (%)", "Intraclass correlation coefficient"])

omnibus_raw.to_csv(TABLES_PATH / "table_experimental_omnibus_raw.csv", index=False)
omnibus_table.to_csv(TABLES_PATH / "table_experimental_omnibus.tsv", sep="\t", index=False)

print("\nPaste-ready (tab separated):\n")
print(omnibus_table.to_csv(sep="\t", index=False))
display(omnibus_table)

### Planned contrasts

In [ ]:
contrast_rows = []
for feature in FEATURES:
    for spec in CONTRASTS:
        d = spec["subset"](df).copy()
        d["contrast_group"] = spec["group"](d)
        d = d.dropna(subset=[feature, "participant_code"])

        levels = sorted(d["contrast_group"].unique())
        if len(levels) < 2:
            print(f"  SKIP [{FEATURE_LABELS[feature]} | {spec['label']}]: "
                  f"only {len(levels)} group present.")
            continue

        ref = spec["reference"]
        md, mdf = fit_mixedlm(
            f"{feature} ~ C(contrast_group, Treatment(reference='{ref}'))", d)
        if mdf is None:
            print(f"  WARNING [{FEATURE_LABELS[feature]} | {spec['label']}]: "
                  f"every optimiser failed; skipping.")
            continue

        keys = [k for k in mdf.params.index if k.startswith("C(contrast_group")]
        if not keys:
            print(f"  WARNING [{FEATURE_LABELS[feature]} | {spec['label']}]: "
                  f"no contrast coefficient found; skipping.")
            continue
        key = keys[0]

        re_var = float(mdf.cov_re.iloc[0, 0])
        resid_var = float(mdf.scale)

        contrast_rows.append({
            "Experiment": EXPERIMENT_LABEL,
            "Feature": FEATURE_LABELS[feature],
            "Contrast": spec["label"],
            "Unstandardised coefficient (b)": float(mdf.params[key]),
            "Standard error": float(mdf.bse[key]),
            "z-statistic": float(mdf.tvalues[key]),
            "p (uncorrected)": float(mdf.pvalues[key]),
            "Random-effect variance": re_var,
            "Residual variance": resid_var,
            "Intraclass correlation coefficient": re_var / (re_var + resid_var),
            "n (trials)": int(len(d)),
            "n (participants)": int(d["participant_code"].nunique()),
        })

contrast_raw = pd.DataFrame(contrast_rows)
if contrast_raw.empty:
    raise ValueError("No contrast could be fitted. Check the CONTRASTS definitions.")

# Correction across every row of the table (features x contrasts).
contrast_raw["p (Holm-adjusted)"] = multipletests(
    contrast_raw["p (uncorrected)"], alpha=0.05, method=CORRECTION_METHOD)[1]
print(f"{CORRECTION_METHOD.title()} correction applied across "
      f"{len(contrast_raw)} test(s) ({contrast_raw['Feature'].nunique()} features "
      f"x {contrast_raw['Contrast'].nunique()} contrasts)")

contrast_raw = contrast_raw[[
    "Experiment", "Feature", "Contrast",
    "Unstandardised coefficient (b)", "Standard error", "z-statistic",
    "p (uncorrected)", "p (Holm-adjusted)",
    "Random-effect variance", "Residual variance",
    "Intraclass correlation coefficient",
    "n (trials)", "n (participants)"]]

contrast_table = format_table(
    contrast_raw,
    sci_cols=["p (uncorrected)", "p (Holm-adjusted)", "Random-effect variance"],
    coef_cols=["Unstandardised coefficient (b)", "Standard error", "z-statistic",
               "Residual variance", "Intraclass correlation coefficient"])

contrast_raw.to_csv(TABLES_PATH / "table_experimental_raw.csv", index=False)
contrast_table.to_csv(TABLES_PATH / "table_experimental.csv", index=False)
contrast_table.to_csv(TABLES_PATH / "table_experimental.tsv", sep="\t", index=False)

print("\nPaste-ready (tab separated):\n")
print(contrast_table.to_csv(sep="\t", index=False))
display(contrast_table)